In [2]:

import torch
from torch import nn
import torch.nn.functional as F
import json
import random
from pathlib import Path


ARC_DIR = Path("/Users/kostas/dev/ARC-AGI/data/training")
REARC_DIR = Path("/Users/kostas/dev/re_arc/tasks")

test_device = torch.device("mps")

In [3]:
class GridEncoder(nn.Module):
    def __init__(self):
        super().__init__()

        self.color_embedding = nn.Embedding(10, 64)
        self.row_embedding = nn.Embedding(39, 64)
        self.column_embedding = nn.Embedding(30, 64)
        self.role_embedding = nn.Embedding(2, 64)

    def forward(self, grid_ids, role_id):
        # grid_ids: [batch, height, width], role_id: 0 = support input, 1 = support output
        
        _, height, width = grid_ids.shape

        rows = torch.arange(height, device=grid_ids.device).view(1, height, 1)

        cols = torch.arange(width, device=grid_ids.device).view(1, 1, width)

        roles = torch.full_like(grid_ids, role_id)

        tokens = (
            self.color_embedding(grid_ids)
            + self.row_embedding(rows)
            + self.column_embedding(cols)
            + self.role_embedding(roles)
        )

        return tokens.flatten(1, 2)  # [batch, height * width, 64]

In [4]:
import json
import random
from pathlib import Path


# Group ARC and RE-ARC files by task ID.
def index_task_files(arc_dir=ARC_DIR, rearc_dir=REARC_DIR):
    index = {}

    for folder in (Path(arc_dir), Path(rearc_dir)):
        for path in sorted(folder.glob("*.json")):
            index.setdefault(path.stem, []).append(path)

    return index



def load_task_pairs(paths):
    pairs = []
    seen = set()

    for path in paths:
        data = json.loads(path.read_text())

        # Use ARC training pairs or the RE-ARC list.
        raw_pairs = data["train"] if isinstance(data, dict) else data

        for pair in raw_pairs:
            key = (
                tuple(map(tuple, pair["input"])),
                tuple(map(tuple, pair["output"])),
            )

            if key not in seen:
                seen.add(key)
                pairs.append({
                    "input": pair["input"],
                    "output": pair["output"],
                })

    return pairs


# seperate training and val tasks 
def split_tasks(index, seed=0, validation_fraction=0.2):
    task_ids = sorted(index)
    random.Random(seed).shuffle(task_ids)

    count = round(len(task_ids) * validation_fraction)

    training_ids = task_ids[count:]
    validation_ids = task_ids[:count]

    return training_ids, validation_ids


# one task episode is 3 support pairs and 1 query pair
def sample_episode(index, task_ids, rng):
    task_id = rng.choice(task_ids)
    pairs = load_task_pairs(index[task_id])

    query = rng.choice(pairs)

    # query input must not be in the support set 
    candidates = [
        pair for pair in pairs
        if pair["input"] != query["input"]
    ]

    support = rng.sample(candidates, min(3, len(candidates)))

    return task_id, support, query

In [5]:
class TransformerBlock(nn.Module):
    def __init__(self):
        super().__init__()

        self.norm1 = nn.LayerNorm(64)

        self.attention = nn.MultiheadAttention(
            embed_dim=64,
            num_heads=4,
            dropout=0.0,
            batch_first=True,
        )

        self.norm2 = nn.LayerNorm(64)

        self.mlp = nn.Sequential(
            nn.Linear(64, 128),
            nn.ReLU(),
            nn.Linear(128, 64),
        )

    def forward(self, x):
        # Normalize the features before attention.
        normalized = self.norm1(x)

        # Each cell token gathers information from other tokens.
        attended, _ = self.attention(
            normalized,
            normalized,
            normalized,
            need_weights=False,
        )

        # Add the gathered information to the existing features.
        x = x + attended

        # Process each token's features and add that result.
        return x + self.mlp(self.norm2(x))

In [6]:
class RuleEncoder(nn.Module):
    def __init__(self):
        super().__init__()

        self.grid_encoder = GridEncoder()

        self.stack = nn.Sequential(
            TransformerBlock(),
            TransformerBlock(),
        )

        self.rule_projection = nn.Linear(64, 64)

    def forward(self, support_pairs):
        device = self.rule_projection.weight.device
        pair_vectors = []

        for pair in support_pairs:
            input_ids = torch.tensor(
                pair["input"],
                dtype=torch.long,
                device=device,
            ).unsqueeze(0)

            output_ids = torch.tensor(
                pair["output"],
                dtype=torch.long,
                device=device,
            ).unsqueeze(0)

            # Join the input and output tokens for this pair.
            embeddings = torch.cat(
                (
                    self.grid_encoder(input_ids, 0),
                    self.grid_encoder(output_ids, 1),
                ),
                dim=1,
            )

            # Repeatedly update the state using the same two blocks.
            state = torch.zeros_like(embeddings)

            for _ in range(3):
                state = self.stack(state + embeddings)

            # Average this pair's tokens into one vector.
            pair_vectors.append(state.mean(dim=1))

        # Average the vectors from all support pairs.
        pooled = torch.stack(pair_vectors, dim=0).mean(dim=0)

        return self.rule_projection(pooled)  # [1, 64]

In [7]:
def encode_query(grid, device):
    ids = torch.tensor(grid, dtype=torch.long, device=device)
    height, width = ids.shape

    one_hot = (
        F.one_hot(ids, num_classes=10)
        .permute(2, 0, 1)
        .float()
        .unsqueeze(0)
    )

    
    padding = (0, 30 - width, 0, 39 - height)
    original = F.pad(one_hot, padding)

    # padding cells = 0 
    valid = F.pad(
        torch.ones(1, 1, height, width, device=device),
        padding,
    )

    sizes = torch.tensor(
        [[height / 39, width / 30]],
        dtype=torch.float32,
        device=device,
    )

    return original, valid, sizes


class RuleNCA(nn.Module):
    def __init__(self):
        super().__init__()

        self.perception = nn.Conv2d(
            in_channels=32,
            out_channels=9 * 32,
            kernel_size=3,
            padding=1,
            groups=32,
            bias=False,
        )

        # Copy each of the nine neighbours into a separate feature.
        selectors = (
            torch.eye(9)
            .reshape(9, 1, 3, 3)
            .repeat(32, 1, 1, 1)
        )

        with torch.no_grad():
            self.perception.weight.copy_(selectors)

        self.perception.weight.requires_grad_(False)

        self.update = nn.Sequential(
            nn.Conv2d(9 * 32 + 10 + 1 + 64, 128, kernel_size=1),
            nn.ReLU(),
            nn.Conv2d(128, 32, kernel_size=1),
        )

        nn.init.normal_(self.update[-1].weight, std=0.001)
        nn.init.zeros_(self.update[-1].bias)

    def forward(self, original, valid, rule):
        batch, _, height, width = original.shape

        #  10 colour , 22 memory .
        memory = original.new_zeros(batch, 22, height, width)
        state = torch.cat((original, memory), dim=1)

        # every cell interaction with the rules(rule vetor)
        rule_map = rule[:, :, None, None].expand(-1, -1, height, width)

        for _ in range(16):
            features = torch.cat((self.perception(state), original, valid, rule_map,), dim=1,)
            change = self.update(features)
            state = state + change

        return state

In [8]:
class Architecture(nn.Module):
    def __init__(self):
        super().__init__()
        #supposed TRM
        self.trm = RuleEncoder()
        #NCA model that takes the rule of TRM 
        self.nca = RuleNCA()

        self.size_features = nn.Sequential(
            nn.Linear(32 + 64 + 2, 128),
            nn.ReLU(),
        )

        self.height_head = nn.Linear(128, 39)
        self.width_head = nn.Linear(128, 30)

    def forward_with_rule(self, query_input, rule):
        original, valid, input_sizes = encode_query( query_input, rule.device)

        state = self.nca(original, valid, rule)

        summary = torch.cat((state.mean(dim=(2, 3)), rule, input_sizes,), dim=1,)
        size_hidden = self.size_features(summary)

        return {
            "colour_logits": state[:, :10],
            "height_logits": self.height_head(size_hidden),
            "width_logits": self.width_head(size_hidden),
        }

    def forward(self, support_pairs, query_input):
        rule = self.trm(support_pairs)
        return self.forward_with_rule(query_input, rule)

In [ ]:
def compute_loss(prediction, target_grid):
    device = prediction["colour_logits"].device

    target = torch.tensor(target_grid, dtype=torch.long, device=device).unsqueeze(0)

    _, height, width = target.shape

    colour_loss = F.cross_entropy(
        prediction["colour_logits"][:, :, :height, :width],
        target,
    )

    height_loss = F.cross_entropy(
        prediction["height_logits"],
        torch.tensor([height - 1], 
        device=device),
    )

    width_loss = F.cross_entropy(
        prediction["width_logits"],
        torch.tensor([width - 1],
        device=device),)
    

    parts = {
        "colour": colour_loss,
        "height": height_loss,
        "width": width_loss,
    }

    return colour_loss + height_loss + width_loss, parts


def training_step(model, optimizer, support, query):
    model.train()
    optimizer.zero_grad(set_to_none=True)
    prediction = model(support, query["input"])
    loss, _ = compute_loss(prediction, query["output"])

    loss.backward()

    # control big gradients
    nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
    optimizer.step()

    return loss.item()


def run_training(steps):
    torch.manual_seed(0)
    rng = random.Random(1)

    index = index_task_files()
    train_ids, val_ids = split_tasks(index, seed=0)

    model = Architecture().to(test_device)

    optimizer = torch.optim.AdamW(model.parameters(),lr=1e-3,weight_decay=1e-4,)

    for step in range(1, steps + 1):
        _, support, query = sample_episode(index, train_ids, rng)

        loss = training_step(model, optimizer, support, query)

        if step == 1 or step % 100 == 0:
            print(f"Step {step} | loss {loss:.4f}", flush=True)

    return model, index, val_ids

In [10]:
def decode_prediction(prediction):
    height = prediction["height_logits"].argmax(dim=1).item() + 1
    width = prediction["width_logits"].argmax(dim=1).item() + 1

    colours = prediction["colour_logits"].argmax(dim=1)[0]

    return colours[:height, :width].detach().cpu().tolist()


@torch.no_grad()
def evaluate(model, index, task_ids, episodes=5, seed=123):#maybe
    was_training = model.training
    model.eval()

    rng = random.Random(seed)

    exact = 0
    correct_cells = 0
    total_cells = 0
    total_loss = 0.0

    for _ in range(episodes):
        _, support, query = sample_episode(index, task_ids, rng)

        prediction = model(support, query["input"])
        loss, _ = compute_loss(prediction, query["output"])
        total_loss += loss.item()

        # check predicted grid with size
        output = decode_prediction(prediction)
        exact += int(output == query["output"])

        # count correct colours 
        target = torch.tensor(query["output"],dtype=torch.long,device=prediction["colour_logits"].device,)
        height, width = target.shape
        colours = prediction["colour_logits"].argmax(dim=1)[0]
        correct_cells += (colours[:height, :width] == target).sum().item()
        total_cells += target.numel()

    model.train(was_training)

    return {
        "episodes": episodes,
        "exact_grid_accuracy": exact / episodes,
        "target_canvas_cell_accuracy": correct_cells / total_cells,
        "mean_loss": total_loss / episodes,
    }

In [11]:
model, index, val_ids = run_training(steps=1000)

report = evaluate(
    model,
    index,
    val_ids,
    episodes=100,
)

print(report)

Step 1 | loss 9.7913
Step 100 | loss 8.1673
Step 200 | loss 7.6690
Step 300 | loss 6.7862
Step 400 | loss 8.1654
Step 500 | loss 7.2930
Step 600 | loss 7.7838
Step 700 | loss 9.6581
Step 800 | loss 10.2154
Step 900 | loss 9.3730
Step 1000 | loss 7.5448
{'episodes': 100, 'exact_grid_accuracy': 0.0, 'target_canvas_cell_accuracy': 0.6377506915629322, 'mean_loss': 8.016407198905945}


In [12]:
%matplotlib inline

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
from matplotlib.patches import Rectangle

# ARC colours, numbered 0–9.
arc_colours = ListedColormap([
    "#000000", "#0074D9", "#FF4136", "#2ECC40", "#FFDC00",
    "#AAAAAA", "#F012BE", "#FF851B", "#7FDBFF", "#870C25",
])

test_model.eval()

fig, axes = plt.subplots(
    len(fixed_queries), 3,
    figsize=(12, 3.5 * len(fixed_queries)),
    squeeze=False,
)

with torch.no_grad():
    for number, query in enumerate(fixed_queries):
        prediction = test_model(support, query["input"])

        input_grid = np.array(query["input"])
        expected = np.array(query["output"])
        predicted = np.array(decode_prediction(prediction))

        grids = [input_grid, expected, predicted]
        titles = [
            f"Example {number + 1}: input",
            "Correct output",
            "Model prediction",
        ]

        for column, (grid, title) in enumerate(zip(grids, titles)):
            ax = axes[number, column]
            height, width = grid.shape

            ax.imshow(
                grid, cmap=arc_colours,
                vmin=0, vmax=9, interpolation="nearest",
            )
            ax.set_title(f"{title} — {height}×{width}")

            # Row and column numbers help locate individual cells.
            ax.set_xticks(range(width))
            ax.set_yticks(range(height))
            ax.tick_params(labelsize=8)

            # Draw cell boundaries.
            ax.set_xticks(np.arange(-0.5, width, 1), minor=True)
            ax.set_yticks(np.arange(-0.5, height, 1), minor=True)
            ax.grid(which="minor", color="#555555", linewidth=0.5)
            ax.tick_params(which="minor", length=0)

        if predicted.shape == expected.shape:
            wrong_positions = np.argwhere(predicted != expected)

            axes[number, 2].set_title(
                f"Prediction — {len(wrong_positions)} wrong cells"
            )

            # Outline corresponding cells in BOTH output grids.
            for row, column in wrong_positions:
                for output_column in (1, 2):
                    axes[number, output_column].add_patch(
                        Rectangle(
                            (column - 0.5, row - 0.5),
                            1, 1,
                            fill=False,
                            edgecolor="red",
                            linewidth=3,
                        )
                    )
        else:
            axes[number, 2].set_title(
                f"Wrong size: {predicted.shape[0]}×{predicted.shape[1]}\n"
                f"Expected: {expected.shape[0]}×{expected.shape[1]}"
            )

plt.tight_layout()
plt.show()

NameError: name 'test_model' is not defined